# 实验六补充 3：扩散模型教学实验

这份 notebook 面向“完全不知道扩散模型是什么”的同学。我们会从一个非常直观的故事开始：先把图片一步步加噪，直到几乎只剩随机噪声；再训练一个模型学习如何反过来去噪。生成图片时，从随机噪声开始，反复去噪，最后得到一张新图片。

默认模型是 `google/ddpm-cifar10-32`。它是 Hugging Face 上的 DDPM 模型，模型卡标注 Apache-2.0，用于生成 32x32 的 CIFAR-10 风格小图像。它不是文本生成图像模型，没有 prompt；它只从随机噪声开始，逐步去噪得到一张小图。

本实验不追求高清图像，而是把扩散模型拆成可观察的步骤。教师版已经填入答案；发布学生版时，`START CODE HERE` 与 `END CODE HERE` 之间的代码会被挖空。整份 notebook 只有 3 处需要学生补代码，其余都是演示。


## 0. 从零理解：什么是扩散模型

扩散模型的训练和生成可以分成两个方向：

1. **正向扩散**：人为规定一个加噪过程，把真实图片 $x_0$ 逐步变成噪声 $x_T$。这一步不需要神经网络。
2. **反向去噪**：训练神经网络学习从 $x_t$ 回到更干净的 $x_{t-1}$。这一步需要 U-Net。

一个很重要的直觉是：扩散模型不是一次性画完整图片，而是从纯噪声开始，经过很多小步逐渐修正。每一步只做一点点去噪。这和语言模型一次生成一个 token 很像：都不是“一口气生成全部”，而是逐步构造结果。

Stable Diffusion 这类大模型在这个基本框架上增加了文本编码器、VAE latent 空间、cross-attention 等模块。本实验故意使用 CIFAR-10 的 32x32 DDPM，先把最基础的扩散过程讲清楚。


## 1. 环境与模型

扩散模型实验需要 `diffusers`。第一次运行会下载模型，之后会缓存在本机。这个模型生成的是 32x32 小图，所以比 Stable Diffusion 轻很多；CPU 也能跑，只是采样会慢一些。

如果缺少依赖，先取消下一格的 `%pip install` 注释并运行。安装后重启 kernel。


In [ ]:
# 如缺少依赖，取消下一行注释并运行一次；安装后重启 kernel。
# %pip install -U diffusers transformers accelerate safetensors matplotlib

import importlib.util
missing = [pkg for pkg in ["torch", "diffusers", "matplotlib"] if importlib.util.find_spec(pkg) is None]
if missing:
    print("缺少依赖：", missing)
    print("请先运行：%pip install -U diffusers transformers accelerate safetensors matplotlib")
else:
    print("依赖检查通过。")


**输出说明**

如果看到“依赖检查通过”，说明当前 kernel 已经有扩散实验所需库。如果提示缺少 `diffusers`，需要先安装，否则无法加载 DDPM pipeline。


In [ ]:
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from diffusers import DDPMPipeline, DDIMScheduler
from PIL import Image

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "google/ddpm-cifar10-32"

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
print("device:", device)


**输出说明**

这里显示本机推理设备。扩散模型需要重复调用 U-Net 多次，所以设备会明显影响速度。后面你改变采样步数时，也会看到步数越多耗时越长。


In [ ]:
pipe = DDPMPipeline.from_pretrained(MODEL_ID).to(device)
pipe.unet.eval()
scheduler = pipe.scheduler

n_params = sum(p.numel() for p in pipe.unet.parameters())
print(f"Loaded {MODEL_ID}")
print(f"UNet parameters: {n_params / 1e6:.1f}M")
print("train timesteps:", scheduler.config.num_train_timesteps)
print("sample size:", pipe.unet.config.sample_size)


**输出说明**

`UNet parameters` 是去噪网络的参数量。`train timesteps` 表示训练时定义了多少个加噪时间步；`sample size` 是生成图片大小。这个模型只生成 32x32 小图，因此很适合课堂上观察扩散过程。


In [ ]:
alpha_bar = scheduler.alphas_cumprod.detach().cpu()
noise_scale = torch.sqrt(1 - alpha_bar)
signal_scale = torch.sqrt(alpha_bar)

plt.figure(figsize=(6, 3))
plt.plot(signal_scale, label="signal scale sqrt(alpha_bar)")
plt.plot(noise_scale, label="noise scale sqrt(1-alpha_bar)")
plt.xlabel("timestep")
plt.ylabel("scale")
plt.title("Forward diffusion schedule")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

for t in [0, 100, 300, 700, 999]:
    print(f"t={t:3d} | signal={float(signal_scale[t]):.3f} | noise={float(noise_scale[t]):.3f}")


**输出说明**

这张曲线展示正向扩散中“原图信号”和“噪声”的比例如何变化。早期 timestep 中信号强、噪声弱；后期 timestep 中噪声几乎占主导。扩散模型训练时会随机抽不同 timestep，让 U-Net 学会处理不同噪声强度。


## 2. 正向扩散：把图片变成噪声

DDPM 的正向过程不是模型学出来的，而是人为规定的加噪过程。给定一张干净图片 $x_0$，在第 $t$ 个时间步，我们可以直接采样得到带噪图片：

$$
x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1-\bar{\alpha}_t}\epsilon, \quad \epsilon \sim \mathcal{N}(0, I)
$$

其中 $\bar{\alpha}_t$ 由 scheduler 的噪声表决定。$t$ 越大，$\bar{\alpha}_t$ 越小，图片中保留的原始信号越少，噪声越强。

这条公式很重要，因为训练时并不需要真的一步一步加噪。我们可以随机抽一个时间步 $t$，直接得到 $x_t$，再让 U-Net 学会从 $x_t$ 和 $t$ 中预测噪声 $\epsilon$。

### 练习 1：手写一次正向加噪

你只需要补 3 行：取出 $\bar{\alpha}_t$，计算信号系数，计算噪声系数。最后的加权求和已经给出。


In [ ]:
def make_toy_image(size=32):
    """构造一张 32x32 的简单彩色图片，用来观察加噪过程。取值范围为 [-1, 1]。"""
    img = torch.zeros(3, size, size)
    yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    img[0] = (xx / (size - 1)) * 2 - 1
    img[1] = (yy / (size - 1)) * 2 - 1
    img[2] = torch.where((xx - size // 2) ** 2 + (yy - size // 2) ** 2 < (size // 4) ** 2, 1.0, -1.0)
    return img.unsqueeze(0)


def manual_add_noise(x0, noise, timestep, scheduler):
    alpha_cumprod = scheduler.alphas_cumprod.to(x0.device)

    ### START CODE HERE ###
    # TODO 1：取出当前 timestep 对应的 alpha_bar。
    # TODO 2：计算干净图像的系数 sqrt(alpha_bar_t)。
    # TODO 3：计算噪声系数 sqrt(1 - alpha_bar_t)。
    ### END CODE HERE ###

    return signal_scale * x0 + noise_scale * noise

x0 = make_toy_image().to(device)
noise = torch.randn_like(x0)
for t in [0, 100, 300, 700, 999]:
    xt = manual_add_noise(x0, noise, t, scheduler)
    print(t, float(xt.mean()), float(xt.std()))


**输出说明**

每一行打印一个 timestep 下带噪图片的均值和标准差。随着 timestep 变大，图像越来越接近标准噪声，标准差通常会接近 1。数字本身不如趋势重要：越往后，原图信息越少。


In [ ]:
def tensor_to_image(x):
    x = x.detach().float().cpu().clamp(-1, 1)
    x = (x + 1) / 2
    arr = (x[0].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return Image.fromarray(arr)

fig, axes = plt.subplots(1, 6, figsize=(14, 3))
axes[0].imshow(tensor_to_image(x0))
axes[0].set_title("clean x0")
axes[0].axis("off")
for ax, t in zip(axes[1:], [0, 100, 300, 700, 999]):
    xt = manual_add_noise(x0, noise, t, scheduler)
    ax.imshow(tensor_to_image(xt))
    ax.set_title(f"t={t}")
    ax.axis("off")
plt.tight_layout()
plt.show()


**输出说明**

这组图最直观地展示了扩散模型的正向过程：清晰图片逐渐被噪声淹没。到很大的 timestep 时，人眼几乎看不出原始图形。反向生成要做的事情，就是从最右边这种噪声状态逐步走回类似左边的清晰图像。


## 3. 训练目标：为什么预测噪声

DDPM 的 U-Net 接收带噪图片 $x_t$ 和时间步 $t$，输出一个和图片同形状的张量。常见训练目标是让输出接近真实噪声 $\epsilon$：

$$
\mathcal{L} = \left\|\epsilon_\theta(x_t, t) - \epsilon\right\|_2^2
$$

直觉上，模型学会了“这张带噪图里哪些部分像噪声”。采样时，我们从纯噪声开始，反复调用 U-Net 预测噪声，再由 scheduler 根据预测结果向更干净的图片走一步。

这里我们不会重新训练扩散模型，只用预训练 U-Net 计算一次噪声预测 loss，帮助你理解训练时每个 batch 在做什么。

### 练习 2：计算一次噪声预测 loss

你只需要补 4 行：随机抽 timestep、加噪、调用 U-Net、计算 MSE loss。


In [ ]:
clean_images = make_toy_image().repeat(2, 1, 1, 1).to(device)
noise = torch.randn_like(clean_images)

### START CODE HERE ###
# TODO 1：为 batch 中每张图随机抽一个 timestep。
# TODO 2：用 scheduler.add_noise 得到 noisy_images。
# TODO 3：把 noisy_images 和 timesteps 喂给 U-Net，取 .sample。
# TODO 4：预测噪声和真实噪声之间做均方误差。
### END CODE HERE ###

print("timesteps:", timesteps.detach().cpu().tolist())
print("noise prediction loss:", float(loss.item()))
assert noise_pred.shape == noise.shape
assert torch.isfinite(loss)


**输出说明**

`timesteps` 是这次训练样本抽到的噪声强度。`noise prediction loss` 衡量 U-Net 预测噪声和真实噪声的差距。真实训练会在大量真实图片、随机 timestep 和随机噪声上重复这个过程，让 U-Net 学到通用去噪能力。


In [ ]:
zero_baseline = F.mse_loss(torch.zeros_like(noise), noise).detach()
model_mse = F.mse_loss(noise_pred.detach(), noise).detach()
print(f"zero predictor MSE: {float(zero_baseline):.4f}")
print(f"pretrained UNet MSE: {float(model_mse):.4f}")
print(f"noise std: {float(noise.std()):.4f} | predicted noise std: {float(noise_pred.detach().std()):.4f}")


**输出说明**

`zero predictor` 是一个完全不会去噪的基线：它永远预测噪声为 0。预训练 U-Net 的 MSE 通常明显更低，说明它确实从带噪图片和 timestep 中提取到了有用信息。`predicted noise std` 也能帮助你判断模型输出是否在合理尺度上。


## 4. 反向采样：从噪声生成图片

反向过程从一个标准高斯噪声图片开始。每一步做三件事：

1. U-Net 根据当前图片 $x_t$ 和时间步 $t$ 预测噪声。
2. scheduler 用预测噪声计算上一个时间步的图片 $x_{t-1}$。
3. 重复直到时间步走到 0。

DDPM 通常需要很多步，质量较稳但慢。DDIM 是一种常用的加速采样方式，可以用更少步数生成图片。课堂上我们用较少步数，重点看流程，而不是追求最佳图像质量。

### 练习 3：手写一个最小反向去噪循环

你只需要补 3 行：调用 U-Net 预测噪声、调用 scheduler.step、更新当前图片。


In [ ]:
@torch.no_grad()
def manual_sample(pipe, num_inference_steps=15, seed=0, batch_size=4, use_ddim=True):
    if use_ddim:
        local_scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    else:
        local_scheduler = pipe.scheduler
    local_scheduler.set_timesteps(num_inference_steps)

    generator = torch.Generator(device=device).manual_seed(seed)
    image = torch.randn(
        batch_size,
        pipe.unet.config.in_channels,
        pipe.unet.config.sample_size,
        pipe.unet.config.sample_size,
        generator=generator,
        device=device,
    )

    for t in local_scheduler.timesteps:
        ### START CODE HERE ###
        # TODO 1：U-Net 预测当前 image 中的噪声。
        # TODO 2：scheduler 根据预测噪声走到上一个时间步。
        # TODO 3：更新 image。
        ### END CODE HERE ###

    return image

samples = manual_sample(pipe, num_inference_steps=12, seed=SEED, batch_size=4, use_ddim=True)
print(samples.shape, float(samples.mean()), float(samples.std()))


**输出说明**

输出形状 `[4, 3, 32, 32]` 表示一次生成了 4 张 RGB 小图。均值和标准差只是粗略检查数值是否正常；真正有趣的是下一格把张量显示成图片。生成质量受步数、seed 和 scheduler 影响很大。


In [ ]:
def show_image_grid(tensors, title=None):
    tensors = tensors.detach().float().cpu().clamp(-1, 1)
    tensors = (tensors + 1) / 2
    n = tensors.shape[0]
    fig, axes = plt.subplots(1, n, figsize=(2.4 * n, 2.4))
    if n == 1:
        axes = [axes]
    for ax, img in zip(axes, tensors):
        ax.imshow(img.permute(1, 2, 0).numpy())
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_image_grid(samples, title="Manual DDIM samples, 12 steps")


**输出说明**

你会看到几张 32x32 的 CIFAR-10 风格小图。它们不会像现代文生图那样高清，但已经能体现扩散模型最有趣的地方：从随机噪声出发，经过少量去噪步骤，出现具有颜色块和物体轮廓的样本。


In [ ]:
@torch.no_grad()
def sample_trace(pipe, num_inference_steps=10, seed=3):
    local_scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    local_scheduler.set_timesteps(num_inference_steps)
    generator = torch.Generator(device=device).manual_seed(seed)
    image = torch.randn(1, 3, 32, 32, generator=generator, device=device)
    trace = [("start", image.detach().clone())]
    keep = {0, num_inference_steps // 3, 2 * num_inference_steps // 3, num_inference_steps - 1}
    for idx, t in enumerate(local_scheduler.timesteps):
        noise_pred = pipe.unet(image, t).sample
        image = local_scheduler.step(noise_pred, t, image).prev_sample
        if idx in keep:
            trace.append((f"step {idx + 1}", image.detach().clone()))
    return trace

trace = sample_trace(pipe, num_inference_steps=10, seed=7)
fig, axes = plt.subplots(1, len(trace), figsize=(2.2 * len(trace), 2.3))
for ax, (name, img) in zip(axes, trace):
    ax.imshow(((img[0].detach().float().cpu().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).numpy())
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()


**输出说明**

这条轨迹展示同一张样本在去噪过程中的变化。开头接近随机噪声，后面逐渐出现更稳定的颜色和结构。真实扩散模型的魅力就在这里：每一步变化很小，但累计起来会从混乱走向有形。


## 5. 小实验：步数、随机种子和 scheduler

扩散模型的输出对随机种子很敏感，因为采样从随机噪声开始。同一个 seed 通常能复现同一张图；换 seed 会得到不同图。

你可以改下面的参数做实验：

- `num_inference_steps`：步数越多通常越稳定，但更慢；步数太少可能图像更糊或更随机。
- `seed`：改变初始噪声，从而改变生成结果。
- `use_ddim`：`True` 使用 DDIM 加速采样；`False` 使用 DDPM scheduler。
- `batch_size`：一次生成几张图；电脑较慢时改成 1 或 2。

下面的演示不需要学生补代码，只是让你感受参数变化的效果。


In [ ]:
experiment_settings = [
    {"num_inference_steps": 8, "seed": 1, "batch_size": 4, "use_ddim": True},
    {"num_inference_steps": 20, "seed": 1, "batch_size": 4, "use_ddim": True},
]

for cfg in experiment_settings:
    start = time.time()
    imgs = manual_sample(pipe, **cfg)
    elapsed = time.time() - start
    print(f"{cfg} | elapsed={elapsed:.2f}s")
    show_image_grid(imgs, title=str(cfg))


**输出说明**

这里比较同一个 seed、不同采样步数的结果。步数更多通常耗时更长，也可能让结构更稳定；但在小模型和很少步数下，质量变化不一定线性。扩散模型工程里经常要在速度和质量之间做权衡。


In [ ]:
same_a = manual_sample(pipe, num_inference_steps=10, seed=123, batch_size=1, use_ddim=True)
same_b = manual_sample(pipe, num_inference_steps=10, seed=123, batch_size=1, use_ddim=True)
diff_c = manual_sample(pipe, num_inference_steps=10, seed=124, batch_size=1, use_ddim=True)
print("same seed max difference:", float((same_a - same_b).abs().max()))
print("different seed mean difference:", float((same_a - diff_c).abs().mean()))
show_image_grid(torch.cat([same_a, same_b, diff_c], dim=0), title="same seed, same seed again, different seed")


**输出说明**

同一个 seed 应该几乎完全复现同一张图；不同 seed 会从不同初始噪声出发，得到不同样本。这说明扩散模型的随机性不是“后期随便加一点扰动”，而是从初始噪声开始就决定了生成路径。


## 6. 总结问题

1. 正向扩散公式中，$\bar{\alpha}_t$ 越小，图片会发生什么变化？
2. 为什么 DDPM 训练时常让模型预测噪声，而不是直接预测干净图片？
3. scheduler 在扩散模型中扮演什么角色？它和 U-Net 的职责有什么不同？
4. 采样步数减少时，速度和图像质量通常如何变化？
5. 这个 CIFAR-10 DDPM 和 Stable Diffusion 有什么共同点？又少了哪些关键模块？
